# 第 1 周末练习解答 —— 技术问答解释器

## 练习目标（理念）

构建一个可复用的小工具：输入技术问题，用清晰解释作答。本笔记本对比两条后端路径：

- **OpenRouter**（OpenAI 兼容 API）上的 `gpt-4o-mini`
- **本地 Ollama**（OpenAI 兼容 `/v1`）上的 `llama3.2:1b`

流式（streaming）路径用于 GPT；Llama 路径为一次性完整响应。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(...)` |
| `messages`（system / user） | system 定「怎么讲」，user 放具体问题 |
| 流式输出 `stream=True` | GPT 路径逐块收集再 `display(Markdown(...))` |
| 多后端同一 SDK | 改 `base_url` 即可切到 OpenRouter 或本地 Ollama |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：`OPENAI_API_KEY`（本练习校验前缀为 `sk-or-v1`，面向 OpenRouter）
3. 若跑 Llama：本机 Ollama 需监听 `http://localhost:11434`，并已拉取 `llama3.2:1b`
4. 在「提问」单元格改写 `question`，再分别跑 GPT 与 Llama 两格对比风格


In [ ]:
# ========== 导入与环境：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 API Key
import os
# 从 openai 导入 OpenAI 客户端类：同一套 SDK 可指向云端或本地兼容端点
from openai import OpenAI
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：在笔记本里漂亮地显示 Markdown（本格主要用 Markdown/display）
from IPython.display import Markdown, display, update_display


# 加载 .env；override=True 表示用文件里的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境变量读取 API Key（名字必须是 OPENAI_API_KEY，与常见课程配置一致）
api_key = os.getenv('OPENAI_API_KEY')

# 简单体检：没有 key / 前缀不对就打印英文提示（文案影响排障流程，保留英文）
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-or-v1"):
    print("An API key was found, but it doesn't start sk-or-v1; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


In [ ]:
# ========== 常量：模型名与两个 API 基址集中写在一处 ==========

# OpenRouter / 云端侧使用的模型 id（字符串必须与平台可用模型一致）
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名（含 :1b 标签；需事先 ollama pull 同名模型）
MODEL_LLAMA = 'llama3.2:1b'

# OpenRouter 的 OpenAI 兼容根地址（后面传给 OpenAI(base_url=...)）
open_router = "https://openrouter.ai/api/v1"
# 本机 Ollama 的 OpenAI 兼容根地址（注意是 /v1，不是原生 /api/chat）
local_llm = "http://localhost:11434/v1"


In [ ]:
# ========== 提示词：system 定教学风格，user 包装具体问题 ==========

# system_prompt：发给模型的角色与回答结构（可运行英文 prompt，勿翻译，改译会改变行为）
system_prompt="""
You are a clear and patient teacher.
Your job is to explain complex topics in a way a smart 12-year-old can understand.
Rules:
- Use simple, everyday language.
- Avoid jargon. If you must use a technical word, define it clearly.
- Use short paragraphs.
- Use at least one real-world analogy.
- Give 3 key takeaways at the end in bullet points.
- Do not assume prior knowledge.
- Be accurate but simple.
- Keep the explanation under 400 words.

Structure your response like this:

1) Simple Definition  
2) How It Works (Step-by-step if needed)  
3) Real-Life Example or Analogy  
4) 3 Key Takeaways
"""

def get_user_prompt(question):
    # 把用户问题嵌进固定英文模板，保证每次提问格式一致
    user_prompt = f"""
    Explain the following in a way a smart 12-year-old would understand: {question}”
    """
    return user_prompt


In [ ]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 练习问题写在三引号字符串里；发给模型的内容保持英文（影响回答，不翻译）
question = """
What is an embedding in LLM Engineering?
"""


In [ ]:
# ========== 路径 A：经 OpenRouter 用 gpt-4o-mini 流式回答 ==========

# 创建指向 OpenRouter 的客户端；密钥用上面读到的 api_key
client = OpenAI(base_url=open_router, api_key=api_key)
# 发起 Chat Completions；stream=True 表示边生成边返回增量块
response = client.chat.completions.create(
    model=MODEL_GPT,
    messages=[
        # system：教学风格与结构
        {"role": "system", "content": system_prompt},
        # user：具体问题（strip 去掉首尾空白，避免多余换行进 prompt）
        {"role": "user", "content": get_user_prompt(question.strip())}
    ],
    stream=True
)

# stream=True 时 response 是可迭代的 chunk；把文本块收集起来再一次显示为 Markdown
result_parts = []
for chunk in response:
    # delta.content 可能为 None（例如纯 role/空增量），有内容才追加
    if chunk.choices[0].delta.content:
        result_parts.append(chunk.choices[0].delta.content)
# 拼成完整字符串，用 Markdown 在笔记本里展示
display(Markdown("".join(result_parts)))


In [ ]:
# ========== 路径 B：经本地 Ollama 用 Llama 3.2 一次性回答 ==========

# 同一个 OpenAI SDK，只改 base_url 指向本机 Ollama 兼容端点
client = OpenAI(base_url=local_llm, api_key=api_key)
# 非流式调用：等整段生成完再取 message.content
response = client.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[
        {"role": "system", "content": system_prompt},
        # 注意：这里原代码未 strip()，与上一格略有差异，逻辑保持原样
        {"role": "user", "content": get_user_prompt(question)}
    ],
)

# 取出助手回复正文，再用 Markdown 展示
result = response.choices[0].message.content
display(Markdown(result))
